# PreProcessing du csv supervise

## Initialisation

In [40]:
import pandas as pd
from skimage.metrics import structural_similarity
import cv2
import numpy as np
import os

In [41]:
df = pd.read_csv('../../kidneyData.csv')

In [42]:
df['path'] = df['path'].str.replace('/content/data/', '')
print(df['path'])

0        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
1        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
2        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
3        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
4        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
                               ...                        
12441    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12442    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12443    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12444    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12445    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
Name: path, Length: 12446, dtype: str


## Suppression des column inutile

In [36]:
df = df.iloc[:, 1:]
df = df.drop('path', axis=1)
df = df.drop('Class', axis=1)
print(df)

            image_id   diag  target
0      Tumor- (1044)  Tumor       3
1        Tumor- (83)  Tumor       3
2       Tumor- (580)  Tumor       3
3      Tumor- (1701)  Tumor       3
4      Tumor- (1220)  Tumor       3
...              ...    ...     ...
12441   Cyst- (2522)   Cyst       0
12442   Cyst- (2627)   Cyst       0
12443    Cyst- (972)   Cyst       0
12444   Cyst- (2323)   Cyst       0
12445   Cyst- (2145)   Cyst       0

[12446 rows x 3 columns]


## Ajout des array dans la column matrice 

Pour faire cela, nous allons passer dans les 4 dossiers et dans chaque image des dossiers.
On va appliquer une redimension de 128 par 128 tout en garddant les proportions.
Nous allons egalement gerer le probleme des couleurs et basculer toute les image en gris.
Une fois cela fait on ajoute l'array numpy de la photo directement dans la ligne correspondante dans l'ordre

In [37]:
# liste de nos 4 dossier images
lst_dossier = ['../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Tumor/', 
               '../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Normal/',
               '../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Stone/', 
               '../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Cyst/']

# taille pour la redimension
taille_standard = (128, 128)

# on creer la column
df["matrice"] = None
# our stocker toutes les array
toutes_les_matrices = []

# on passe dans chaque dossier
for dossier in lst_dossier:
    # boucle pour parcourir tout le dossier
    for i, nom_fichier in enumerate(os.listdir(dossier)):
        # on combimne le chemin avec le nom du fichier pour avoir le bon link
        chemin_complet = os.path.join(dossier, nom_fichier)
        
        # lecture de l'img
        img = cv2.imread(chemin_complet)
        
        # on passe en gris les images pour regler les problemes de rgb
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        #redimension
        img_resized = cv2.resize(gray, taille_standard)

        # on ajoute dans notre iste d'array l'array qu'on vient de recuperer
        toutes_les_matrices.append(img_resized.flatten())

        

# on stocke dans matrice l'array de toute les images
df["matrice"] = toutes_les_matrices
print(df['matrice'])


0        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                               ...                        
12441    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
12442    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
12443    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
12444    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
12445    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: matrice, Length: 12446, dtype: object


## Export de notre dataset en CSV

In [38]:
df.to_csv("../../Supervise.csv")